In [1]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests
import json
import hashlib
import glob

c:\Users\zyzai\miniconda3\envs\diss_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None
    return text or extract_text_with_unstructured(file_path) or ""


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n• ", "\n- ", "\n", ". ", " ", ""]
)

base_path = Path("Diss_doc")
all_chunks = []

for folder in sorted(p for p in base_path.iterdir() if p.is_dir()):
    for file in sorted(folder.rglob("*")):
        if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
            raw_text = extract_text(file)
            if not raw_text.strip():
                continue

            chunks = text_splitter.split_text(raw_text)
            for idx, chunk in enumerate(chunks):
                if not chunk.strip():
                    continue
                all_chunks.append({
                    "content": chunk,
                    "source": f"{folder.name}/{file.name}",
                    "doc_type": file.suffix.lower().lstrip("."),
                    "folder": folder.name,
                    "filename": file.name,
                    "chunk_idx": idx,
                })


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = all_chunks  # no need to copy

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")

dim = embeddings.shape[1]
index = faiss.IndexIDMap2(faiss.IndexFlatIP(dim))
ids = np.arange(len(embeddings), dtype=np.int64)
index.add_with_ids(embeddings, ids)

def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vec, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        results.append({
            "score": float(scores[0][i]),
            "source": metas[idx]["source"],
            "chunk_idx": metas[idx]["chunk_idx"],
            "doc_type": metas[idx]["doc_type"],
            "preview": texts[idx][:500],
        })
    return results


Batches: 100%|██████████| 33/33 [00:03<00:00, 10.94it/s]


In [5]:
persist_dir = Path("vector_store")
persist_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(persist_dir / "docs.index"))
with open(persist_dir / "metas.json", "w", encoding="utf-8") as f:
    json.dump(metas, f, ensure_ascii=False)
np.save(persist_dir / "texts.npy", np.array(texts, dtype=object))

In [6]:
def query_mistral(prompt, model="mistral:latest"):
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return r.json()["response"]


In [7]:
def build_prompt(query, context_chunks_with_sources, mode="initial", history_snippets=None):
    # keep last few turns if history exists
    conversation = "\n".join(history_snippets[-5:]) if history_snippets else ""

    # context from retrieved docs
    context_text = "\n".join(f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources) or "N/A"

    if mode == "initial":
        return f"""
You are given excerpts from past project documents. Use ONLY these excerpts.

--- CONTEXT ---
{context_text}
--- END CONTEXT ---

Conversation so far:
{conversation if conversation else "N/A"}

Task:
- List stated client requirements (not inferred).
- Find similar past projects (cite [filename]).
- For each: give a short factual summary, tools, industries.
- Output in bullet list format only.
"""
    else:  # follow-up
        return f"""
You are continuing a consulting discussion.

--- CONTEXT ---
{context_text}
--- END CONTEXT ---

Conversation so far:
{conversation if conversation else "N/A"}

Follow-up request:
\"\"\"{query}\"\"\"

Task:
- Resolve references using conversation.
- Use bullet points, cite [filename] inline.
- If unsupported, answer "N/A".
"""


In [8]:
def _get_source_label(meta):
    if isinstance(meta, dict):
        return meta.get("source", meta.get("filename", "unknown"))
    return str(meta)

def rag(query, top_k=10, min_score=0.5, mode="initial", history_snippets=None):
    # ✅ reuse earlier search()
    results = search(query, top_k=top_k)

    pairs, ranked = [], []
    for r in results:
        if r["score"] < min_score:
            continue
        pairs.append((r["preview"], r["source"]))
        ranked.append((r["score"], r["source"]))

    if not pairs and not history_snippets:
        return None

    if not pairs:
        print("\n no Matches Above Threshold:")
        for i, (s, src) in enumerate(ranked, 1):
            print(f"{i:>2}. {src}  |  score={s:.4f}")

    prompt = build_prompt(
        query,
        pairs,
        history_snippets=history_snippets,
        mode=mode,
    )
    resp = query_mistral(prompt)

    print("\n LLM Response:")
    print(resp)
    return resp, pairs, ranked


In [9]:
def _delete_faiss(path_prefix: str):
    for path in glob.glob(path_prefix + "*"):
        try:
            os.remove(path)
            print("DBs deleted")
        except FileNotFoundError:
            pass


def run_chatbot(top_k_initial=30, top_k_followup=12, min_score=0.5):
    mode = "initial"
    history = []  # ✅ structured history

    while True:
        user_q = input("\nYou: ").strip()
        if not user_q:
            continue
        if user_q.lower() in {"/quit", "exit", "quit"}:
            _delete_faiss(str(persist_dir / "docs.index"))
            print("Bye.")
            break

        tk = top_k_initial if mode == "initial" else top_k_followup
        
        result = rag(
            user_q,
            top_k=tk,
            min_score=min_score,
            mode=mode,
            history_snippets=[h["response"] for h in history] if mode == "followup" else None,
        )
        if not result:
            continue
        resp, _, _ = result

        
        history.append({
            "query": user_q,
            "response": resp,
            "facts": {}   
        })

        mode = "followup"



run_chatbot()



 LLM Response:
 - Client Requirements:
  - Support for hybrid working (Clarasys - Pilgrims_ Friend Society_ Discovery proposal)
  - Transformation of internal services to a consolidated and aligned service based on streamlined operations (St John Ambulance project)
  - Customer centric approach, digital mindset, integrated Microsoft architecture, portfolio of work aligned with strategic business objectives, user-centered approach for change delivery (CX Discovery and Transformation Office project)
  - Experience in user research and service design across various sectors including Healthcare & Public Sector, Media & Information Services, Financial & Professional Services, Not-for-profit, and Retail (User Centred Design & User Research Sales Deck)
  - Cost Optimisation and Savings Discovery (Informa Procurement Technology Cost Optimisation project)

- Past Projects:
  - Clarasys provided local support and oversight by deploying a hybrid working model tailored to MAG’s needs. The team me

NameError: name 'os' is not defined